We compare synthesize data from following aspects:

- Utility: application to downstream task
    - Machine learning efficiency
    - Missing value imputation
- Fidelity: how realistic is synthetic data
    - Low-order statistics
    - High-order statistics
    - Real vs synthetic detection
- Privacy protection
    - Distance to closest record

In [1]:
from pathlib import Path
import os

# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")
set_project_root()

from logging import INFO
from typing import Any

from hydra import initialize, compose
import pandas as pd
from omegaconf import DictConfig, OmegaConf
import json

from midst_toolkit.common.logger import log
from midst_toolkit.data_processing.midst_data_processing import load_midst_data_with_test
from midst_toolkit.evaluation.metrics_base import MetricBase

from midst_toolkit.evaluation.privacy.distance_preprocess import preprocess_for_distance_computation
from midst_toolkit.evaluation.privacy.distance_utils import NormType
from midst_toolkit.evaluation.privacy.epsilon_identifiability_risk import EpsilonIdentifiabilityNorm
from midst_toolkit.evaluation.quality import (
    AlphaPrecision,
    CorrelationMatrixDifference,
    KolmogorovSmirnovAndTotalVariation,
    MeanConfidenceIntervalOverlap,
    MeanF1ScoreDifference,
    MeanRegressionDifference,
)
from midst_toolkit.evaluation.quality.confidence_interval_overlap import ConfidenceLevel

# Local imports
from implementations.tabular_data.evaluation.preprocessing import (   
    get_numerical_and_categorical_column_names,
    preprocess_data_for_alpha_precision_eval,
    syntheval_preprocess,
)
from implementations.tabular_data.evaluation.display_utils import log_metrics

/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the data for quality, utility and privacy evaluation
Evaluation also needs a JSON file containing meta information about the dataset. The provided `meta_info.json` JSON file should provide information about the columns as well as the target column, specifically which columns correspond to numerical and categorical values, which column corresponds to a label (if any), and the downstream task to be performed with this dataset.
The provided pre-processing script (`preprocess_berka_trans.py`), generates this JSON file for the transaction table in the Berka dataset, and the evaluation pipeline loads it as a dictionary. Here is an example of `meta_info`.
```bash
}
    "num_col_idx": [0,3,4,7],
    "cat_col_idx": [1,2,5,6],
    "target_col_idx": [1],
    "task_type": "multiclass"
}
```
Types of supported tasks: "binclass","multiclass", and "regression"

Other dataframes to load:

- `real_train_data`: real data that is used to train the generative model

- `real_holdout_data`: real data that is NOT used in training

- `synthetic_data`: the synthesized data 



In [3]:
ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data" / "single_table" 
# Set data and output directories
base_data_dir = IMPLEMENTATION_ROOT / "data"
base_output_dir = IMPLEMENTATION_ROOT / "results"

TABLE_NAME = "trans"

real_train_data = pd.read_csv(base_data_dir / f"{TABLE_NAME}.csv")
real_holdout_data = pd.read_csv(base_data_dir / f"{TABLE_NAME}_holdout.csv")
synthetic_data = pd.read_csv(base_output_dir / "single_table_synthesizing/trans/_final"/ f"{TABLE_NAME}_synthetic.csv")

with open(base_data_dir / "meta_info.json", "r") as f:
    meta_info = json.load(f)

log(INFO, f"Loaded {TABLE_NAME} data for evaluations")
log(INFO, f"Loaded meta_info for {TABLE_NAME} data")
log(INFO, f"Loaded {len(real_train_data)} rows of real training data")
log(INFO, f"Loaded {len(real_holdout_data)} rows of real holdout data")
log(INFO, f"Loaded {len(synthetic_data)} rows of synthetic data")


INFO :      Loaded trans data for evaluations
INFO :      Loaded meta_info for trans data
INFO :      Loaded 16000 rows of real training data
INFO :      Loaded 4000 rows of real holdout data
INFO :      Loaded 3200 rows of synthetic data


In [3]:
meta_info

{'num_col_idx': [0, 3, 4, 7],
 'cat_col_idx': [1, 2, 5, 6],
 'target_col_idx': [1],
 'task_type': 'multiclass'}

## Pre-process the data for evaluation

In [5]:
# Shared preprocessing for syntheval based metrics if they are to be run
log(INFO, "Preprocessing Data with SynthEval pipeline")
numerical_columns, categorical_columns = get_numerical_and_categorical_column_names(real_train_data, meta_info)
# Categorical values are ordinal encoded, numerical values are min-max scaled
syntheval_real_data_train, syntheval_synthetic_data, syntheval_real_data_holdout = syntheval_preprocess(
    numerical_columns, categorical_columns, real_train_data, synthetic_data, real_holdout_data
)

INFO :      Preprocessing Data with SynthEval pipeline


## Fidelity Evaluations

### Fidelity metric: $\alpha$-Precision and $\beta$-Recall
Compute several quality metrics based on the Alpha Precision measure originally proposed in
https://arxiv.org/abs/2301.07573 comparing the quality of synthetically generated data to real data.
Note that these metrics can be evaluated for each synthetic data point (which are useful for auditing and post-processing). Here we average the scores to reflect the overall quality of the data.




$\alpha$-Preicison and $\beta$-Recall are generalizations of Precision and Recall metrics proposed by [Sajjadi et al.](https://proceedings.neurips.cc/paper/2018/hash/f7696a9b362ac5a51c3dc8f098b73923-Abstract.html) in 2018. These metrics can range between $[0, 1]$ and the closest they are to $1$ the better.


- `Preicison`: measures the **fidelity or quality** of sythetic data. In more clear terms, it computes the proporation of synthetic datapoints that are *close* to real datapoints.  
- `Recall`: measures the **diversity** of synthetic data; i.e. the extent to which these samples cover the full variability of real samples. More clearly, recall computes the proportion of real datapoints that are *close* to synthetic datapoints.  


If we denote the distibution of real datapoints by $P(X)$ and the distribution of sythetic datapoints by $Q(Y)$, precision is the portion of $Q(Y)$ that can be generated by $P(X)$, while recall is the portion of $P(X)$ that can be generated by $Q(Y)$.


To better understand these concepts, let's assume that the real/generated dataponits are samples from an underlying real/generated manifold.
Precision measures the proportation of generated datapoints that fall on the real manifold, while recall measures the proportion of real datapoints that fall on the generated manifold.



**Outliers**: The Precision-Recall metrics are very sensitive to outliers since even a few outliers can greatly change the shape of the underlying manifold.
To address this limitation, $\pmb{\alpha}$**-Precision** and $\pmb{\beta}$**-Recall** are defined by assuming that a fraction $1−\alpha$ (or $1−\beta$) of the real (and synthetic) data are “outliers”, and $\alpha$ (or $\beta$) are “typical”. 
$\alpha$-Precision is the fraction of synthetic samples that resemble the “most typical” fraction $\alpha$ of real samples, whereas $\beta$-Recall is the fraction of real samples covered by the most typical fraction $\beta$ of synthetic samples.
The two metrics are evaluated for all $\alpha, \beta \in [0, 1]$, providing entire precision and recall curves instead of single numbers.


To illustrate, consider the below image. Blue and red points are real and generated datapoints, respectively. The large blue and red spheres show the underlying manifold that was estimated from real and generated datapoints.
Good quality generated datapoints should fall within the blue sphere like image *(c)*. They should not lie far from the blue sphere like *(a)*. Moreover, they should not be placed too close (or *copied*) to a real datapoint like *(b)*.
Image *(d)* shows an outlier in the real datapoints which is cut outside of the manifold due to the application on $\alpha$ and $\beta$.
If we used vanilla Precision and Recall, the blue sphere's radius should have increased to include the outlier which would also lead it to include noisy synthetic datapoints like *(a)*.

<div align="center">
  <img src="./images/alpha-precision.png" alt="alpha-precision image" width="530" height="400">
</div>

An example of low diversity but high quality measure could be a generated dataset manifold that is fully contained inside the real data manifold.
<div align="center">
        <img src="./images/low_diversity_space.png" alt="Low diversity space" width="320" height="200">
</div>


In [3]:
# The implementation is based heavily on the Synthcity library (https://github.com/vanderschaarlab/synthcity).
# Specifically, this class computes the alpha-precision, beta-recall, and authenticity scores between the two datasets. 

# Step 1
log(INFO, "Preprocessing Data for Alpha Precision Evaluation")
# Categorical values are one-hot encoded, numerical values are left alone.
alpha_precision_real_data, alpha_precision_synthetic_data = preprocess_data_for_alpha_precision_eval(
    real_data=real_train_data, synthetic_data=synthetic_data, meta_info=meta_info
)
#NOTE: Synthcity requires that the real and synthetic dataframes have the SAME number of datapoints.
if len(alpha_precision_real_data) > len(alpha_precision_synthetic_data):
    alpha_precision_real_data = alpha_precision_real_data.iloc[:len(alpha_precision_synthetic_data)]
    log(INFO, f"Truncated real data to {len(alpha_precision_synthetic_data)} rows")
else:
    alpha_precision_synthetic_data = alpha_precision_synthetic_data.iloc[:len(alpha_precision_real_data)]
    log(INFO, f"Truncated synthetic data to {len(alpha_precision_real_data)} rows")

# Step 2
log(INFO, "Running Alpha-Precision Evaluation")
metric = AlphaPrecision(naive_only=False)
results = metric.compute(alpha_precision_real_data, alpha_precision_synthetic_data)

INFO :      Preprocessing Data for Alpha Precision Evaluation
INFO :      Truncated real data to 3200 rows
INFO :      Running Alpha-Precision Evaluation
/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/midst_toolkit/evaluation/quality/synthcity/one_class.py:158: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.X = torch.tensor(x_train.reshape((-1, self.input_dim))).float()
INFO :      delta_precision_alpha_OC: 0.26908333333333323
INFO :      delta_coverage_beta_OC: 0.3721458333333333
INFO :      authenticity_OC: 0.991875
INFO :      delta_precision_alpha_naive: 0.22668749999999993
INFO :      delta_coverage_beta_naive: 0.0006041666666667389
INFO :      authenticity_naive: 0.9996875


## Fidelity: Density Estimation of columns

### Single-column similarity score
Essentially, each column in a table represents a single feature which accepts various values from a certain type; e.g. numerical, categorical, datetime or boolean. This feature can be described as a random variable where the values listed under the column are its samples. As a result, the density distribution of each column can be computed and compared between the real data and the synthetic data.


The better the distributions match each other, the better the quality of synthetic data.

The similarity of the two distriutions can be measured via different metrics for different data types; for example, **KSComplement (Kolmogorov-Smirnov Complement)** is used for numerical features and **TVComplement (Total Variation Distance Complement)** for categorical features. 


Both are performed as two-sided hypothesis tests to determine whether it
is likely that the distribution of a given column is the same between the two dataframes (null).
The main score is the average test statistic across all evaluated columns. Smaller is better. 


### Details:

#### **KST (Kolmogorov-Smirnov Test)**  
Computes the cumulative distribution function (CDF) of a numerical random variable for real and synthetic data. Then, finds the maximum different between the two CDFs $M$. Finally, the *KSComplement* score is defined as $1 - M$ so that the higher the score, the more similar the distributions.


The graphs below show two examples with real and synthetic data (black and green). At the left, the synthetic data is similar to the real data so the score is close to 1. At the right, the shapes are different so the score is lower.

<div align="center">
  <img src="./images/KSComplement.png" alt="KST (Kolmogorov-Smirnov Test)" width="600" height="200">
</div>


Source: https://docs.sdv.dev/sdmetrics/data-metrics/quality/kscomplement




**TVD (Total Variation Distance)**  
Computes the frequency of each category's appearance under a certain column and defines it as the said categorie's probability. Then, it computes the sum of differences of probabilities between real and synthetic data as 
$\delta(R, S) = \frac{1}{2} \sum_{\omega \in \Omega}  | R_\omega - S_\omega |$
.

Here, $\omega$ describes all the possible categories in a column, $\Omega$. Meanwhile, $R$ and $S$ refer to the real and synthetic frequencies for those categories. The *TVComplement* returns $1-TVD$ so that a higher score means higher quality.

The bar graph below compares real and synthetic data. Because of the differences between the categories, the TVComplement score is less than 1.0.

<div align="center">
  <img src="./images/TVD.png" alt="TVD (Total Variation Distance)" width="600" height="200">
</div>


### Metric results:
`KolmogorovSmirnovAndTotalVariation.compute()` compares each column in real vs synthetic data with a two-sided test of “same distribution?” Numerical columns use a Kolmogorov–Smirnov (KS) test; categorical columns use Total Variation Distance (TVD) plus a permutation test for the p-value.

**Important:** these are *raw distances*, not the KSComplement / TVComplement scores above. Here **smaller is better** (0 = identical distributions). Complement scores are `1 - distance`.

To compute each return key: the metric first computes the distance of **every column**, then averages those distances.

- **`avg stat` / `stat err`**: Mean (and standard error) of all column-level distances — KS for numbers and TVD for categories mixed together. This is the headline “how far apart are the univariate distributions?” number.
- **`avg ks` / `ks err`**: Same idea, but **only numerical columns**. KS is the largest vertical gap between the two CDFs (the $M$ in the figure above).
- **`avg tvd` / `tvd err`**: Same idea, but **only categorical columns**. TVD is $\frac{1}{2}\sum |R_\omega - S_\omega|$: how much the category frequencies disagree.
- **`avg pval` / `pval err`**: Mean (and SE) of the per-column p-values. A **small p-value** means “this gap is unlikely if the two columns really came from the same distribution.” Averaging p-values is a rough summary; the next keys are easier to interpret.
- **`num sigs` / `frac sigs`**: Count and fraction of columns whose p-value is below `significance_level` (default 0.05) — i.e. columns we treat as **significantly different**. `sigs cols` lists those column names.

Standard error (`* err`) is not “measurement noise”; it is how much the per-column scores vary around their mean. A large `stat err` means some columns match well and others do not.

In [5]:
log(INFO, "Running Kolmogorov-Smirnov and Total Variation Evaluation")
metric = KolmogorovSmirnovAndTotalVariation(
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
    significance_level=0.05,
    permutations=10,
    # Already preprocessing above
    do_preprocess=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data)
results

INFO :      Running Kolmogorov-Smirnov and Total Variation Evaluation


{'avg stat': 0.48472187499999997,
 'stat err': 0.06662868216967383,
 'avg ks': 0.4980937499999999,
 'ks err': 0.025792320855327604,
 'avg tvd': 0.47135000000000005,
 'tvd err': 0.14118308739600033,
 'avg pval': 0.045454545454545456,
 'pval err': 0.017180203318601234,
 'num sigs': 4,
 'frac sigs': 0.5,
 'sigs cols': ['trans_date', 'amount', 'balance', 'account']}

## Fidelity: Confidence Interval Overlap (CIO)

While KS/TVD compare whole column distributions, this metric asks a narrower question: for each **numerical** column, is the **mean** of the synthetic data compatible with the mean of the real data? Compatibility is measured by how much the confidence intervals (CIs) around the two means overlap. Categorical columns are ignored.

**More overlap is better**: 1 means the intervals coincide, 0 means they do not touch at all.

Each CI is built around the column mean using the standard error of the mean and a **z**-score (not a t-score):

$$\text{CI} = \bar{x} \pm z \cdot \frac{s}{\sqrt{n}}, \qquad z = 1.96 \ \text{at}\ 95\%$$

where $\bar{x}$ is the column mean, $s$ its standard deviation, and $n$ the number of rows. Since the interval shrinks as $n$ grows, on large datasets even a small shift in the mean can drive the overlap to 0.

### Per-column overlap score

For the real CI $[L_r, U_r]$ and synthetic CI $[L_s, U_s]$, the shared width is $w = \min(U_r, U_s) - \max(L_r, L_s)$. If $w \le 0$ the intervals miss and the score is 0. Otherwise

$$J = \frac{1}{2}\left(\frac{w}{U_r - L_r} + \frac{w}{U_s - L_s}\right)$$

which averages the fraction of *each* interval that is shared, so a wide interval is not rewarded just for being wide. Example: real $[1, 3]$ and synthetic $[2, 5]$ share $w = 1$, giving $J = \frac{1}{2}(1/2 + 1/3) \approx 0.42$.

### Metric results

- **`avg overlap`**: mean of $J$ over all numerical columns, in $[0, 1]$. Closer to 1 is better.
- **`overlap err`**: standard error of those per-column scores; a large value means some columns match and others do not.
- **`num non-overlaps`**: number of columns with $J = 0$.
- **`frac non-overlaps`**: fraction of columns with no overlap. Closer to 0 is better.

`avg overlap` of 0 with `frac non-overlaps` of 1 means every numerical column's synthetic mean falls outside the real mean's CI.

In [ ]:
log(INFO, "Running Confidence Interval Overlap Evaluation")
metric = MeanConfidenceIntervalOverlap(
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
    confidence_level=ConfidenceLevel(95),
    # Already preprocessing above
    do_preprocess=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data)
results

INFO :      Running Confidence Interval Overlap Evaluation


{'avg overlap': 0.0,
 'overlap err': 0.0,
 'num non-overlaps': 4.0,
 'frac non-overlaps': 1.0}

## Fidelity: Mean Correlation Matrix Difference (Pair-wise Correlation)

KS/TVD and CIO check **one column at a time**. This metric checks **pairs of columns**: do the same relationships show up in synthetic data (e.g. does `amount` still rise with `balance`)?

Build a correlation matrix for real data and one for synthetic data. Each entry is “how strongly column $i$ is associated with column $j$.” Subtract the two matrices and take the **Frobenius norm** of the difference — the square root of the sum of squared cell-wise errors. **Closer to 0 is better** (identical pairwise structure). The score grows with the number of columns, so compare it only across runs with the same `corr_mat_dims`.

### How each pair is scored

With `compute_mixed_correlations=True` (as below), every pair of columns is included. Categories can stay as labels; they do not need one-hot encoding.

- **Number vs number** — Pearson $r \in [-1, 1]$: linear co-movement.
- **Category vs category** — Cramér’s $V \in [0, 1]$: association from a contingency table (0 = independent, 1 = one category determines the other).
- **Category vs number** — correlation ratio $\eta \in [0, 1]$: how much of the numeric column’s variance is explained by the category (e.g. does `type` predict `amount`?).

If `compute_mixed_correlations=False`, only the numerical–numerical block is used.

### Metric results

- **`corr_mat_diff`**: Frobenius distance between the two matrices. 0 is a perfect match.
- **`corr_mat_dims`**: matrix size (number of columns included). With mixed correlations this is all columns (here 8).


In [12]:
log(INFO, "Running Mean Correlation Matrix Difference Evaluation")
metric = CorrelationMatrixDifference(
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
    compute_mixed_correlations=True, # Whether or not to compute correlations between the categorical variables and
    # the categorical and numerical variables. 
    do_preprocess=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data)
results

INFO :      Running Mean Correlation Matrix Difference Evaluation


{'corr_mat_diff': 2.121534424662767, 'corr_mat_dims': 8}

## Utility: F1 Score Difference

Fidelity asks whether synthetic data *looks* like real data. **Utility** asks whether it is *useful*: can you train a classifier on synthetic rows and still predict a real label well?

Here the label is `trans_type` (multiclass). Four models are trained twice — random forest, AdaBoost, SVM, and logistic regression — once on **real** data and once on **synthetic** data. Both versions are then scored with **macro F1** on **real** test rows.

- **Train on Real, Test on Real (TRTR)** baseline: how well a model trained on real data does.
- **Train on Synthetic, Test on Real (TSTR)** utility: how well a model trained on synthetic data does.

The score for each model is $\text{TSTR} - \text{TRTR}$. **Closer to 0 is better.** Negative means synthetic training is worse; positive (rare) means it is better.

Two evaluations:

1. **Cross-validation** — 5 folds of the real training set. Each fold trains on real vs synthetic and tests on the held-out real fold.
2. **Holdout** — train on the *full* real and synthetic sets (no CV), test on `real_holdout_data`. This is the stricter check.

Categorical columns (including the label) must be encoded first. This notebook already ran the SynthEval preprocessor (ordinal encode + min-max scale), so `do_preprocess=False`. The label is passed separately and is **not** listed in `numerical_columns` / `categorical_columns`.

### Metric results

Headline numbers (average over the four models):

- **`mean_f1_difference`**: mean of $(\text{TSTR} - \text{TRTR})$ from CV. Near 0 is best.
- **`f1_difference_standard_error`**: how much that gap varies across models.
- **`mean_f1_difference_holdout`** / **`f1_difference_standard_error_holdout`**: same, but on the holdout set.

Per-model F1 (higher is better for each number; the *gap* is what matters):

- `*_real_train_f1` — TRTR on CV folds
- `*_synthetic_train_f1` — TSTR on CV folds
- `*_real_train_f1_holdout` / `*_synthetic_train_f1_holdout` — same on holdout



In [6]:
from midst_toolkit.common.enumerations import TaskType
from implementations.tabular_data.evaluation.preprocessing import remove_label_column_from_other_columns

TASK_TYPE = "multiclass"
LABEL_COLUMN = "trans_type"
# Explicitly removing the target/label column from other column names
filtered_numerical_columns, filtered_categorical_columns = remove_label_column_from_other_columns(
    LABEL_COLUMN, numerical_columns, categorical_columns
)
assert TASK_TYPE != TaskType.REGRESSION, "F1 Score Difference is only supported for classification tasks"
log(INFO, "Running F1 Score Difference Evaluation")
metric = MeanF1ScoreDifference(
    categorical_columns=filtered_categorical_columns,
    numerical_columns=filtered_numerical_columns,
    label_column=LABEL_COLUMN,
    folds=5,
    f1_type="macro",
    # Already preprocessing above
    do_preprocess=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data, syntheval_real_data_holdout)
log_metrics("F1 Score Difference", results)

INFO :      Running F1 Score Difference Evaluation
INFO :      
F1 Score Difference
--------------------------------------------------------------------------------

INFO :      Metric: random_forest_real_train_f1\tScore: 0.8078533176364469
INFO :      Metric: random_forest_synthetic_train_f1\tScore: 0.346830479339317
INFO :      Metric: adaboost_real_train_f1\tScore: 0.6442279345908849
INFO :      Metric: adaboost_synthetic_train_f1\tScore: 0.24933160927054243
INFO :      Metric: svm_real_train_f1\tScore: 0.661935760004855
INFO :      Metric: svm_synthetic_train_f1\tScore: 0.18555770660827467
INFO :      Metric: logreg_real_train_f1\tScore: 0.5842187408567382
INFO :      Metric: logreg_synthetic_train_f1\tScore: 0.18555770660827467
INFO :      Metric: mean_f1_difference\tScore: -0.43273956281562903
INFO :      Metric: f1_difference_standard_error\tScore: 0.006733909472004676
INFO :      Metric: random_forest_real_train_f1_holdout\tScore: 0.6627851549199865
INFO :      Metric: random_f

## Utility: Regression Score Difference

In [7]:
# Before running the next cell, make sure to add the previous label column back to the categorical columns
if "trans_type" not in categorical_columns: categorical_columns.append("trans_type")
# Remove any repeated columns
categorical_columns = list(set(categorical_columns))
numerical_columns = list(set(numerical_columns))
print(categorical_columns)
print(numerical_columns)

['k_symbol', 'operation', 'trans_type', 'bank']
['account', 'trans_date', 'balance', 'amount']


In [9]:
from midst_toolkit.common.enumerations import TaskType
from implementations.tabular_data.evaluation.preprocessing import remove_label_column_from_other_columns
TASK_TYPE = "regression"
# Make sure that the label column is a numerical column
LABEL_COLUMN = "balance"
# Specify the regression models' parameters and structure in a config file. Make sure it's a PATH object.
REGRESSOIN_CONFIG_PATH = Path("implementations/tabular_data/evaluation/regression_config.yaml")
# Explicitly removing the target/label column from other column names
filtered_numerical_columns, filtered_categorical_columns = remove_label_column_from_other_columns(
    LABEL_COLUMN, numerical_columns, categorical_columns
)
log(INFO, "Running Regression Score Difference Evaluation")
metric = MeanRegressionDifference(
    categorical_columns=filtered_categorical_columns,
    numerical_columns=filtered_numerical_columns,
    label_column=LABEL_COLUMN,
    preprocess_labels=True,
    include_additional_metrics=True,
    # Regression has it's own preprocessing pipeline
    do_preprocess=True,
    regressors_config = REGRESSOIN_CONFIG_PATH,
    measure_metrics_in_original_label_space=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data, syntheval_real_data_holdout)
results

INFO :      Running Regression Score Difference Evaluation
INFO :      Default preprocessing will be performed during computation.
100%|██████████| 1/1 [00:00<00:00, 45.01it/s]
INFO :      Getting Regressor: LinearRegression
INFO :      Getting Regressor: LinearRegression
INFO :      Getting Regressor: LinearRegression
INFO :      Getting Regressor: LinearRegression
100%|██████████| 12/12 [00:15<00:00,  1.27s/it]
INFO :      Getting Regressor: MLPRegressor
INFO :      Getting Regressor: MLPRegressor
INFO :      Getting Regressor: MLPRegressor
INFO :      Getting Regressor: MLPRegressor
  0%|          | 0/36 [00:00<?, ?it/s]INFO :      Getting Regressor: XGBRegressor
INFO :      Getting Regressor: XGBRegressor
  6%|▌         | 2/36 [00:00<00:02, 12.15it/s]INFO :      Getting Regressor: XGBRegressor
INFO :      Getting Regressor: XGBRegressor
 11%|█         | 4/36 [00:00<00:02, 13.29it/s]INFO :      Getting Regressor: XGBRegressor
INFO :      Getting Regressor: XGBRegressor
 17%|█▋      

{'LinearRegression_r2_difference': -5.197439136207873,
 'LinearRegression_explained_variance_difference': -4.978378701907841,
 'LinearRegression_mean_squared_error_difference': 0.05623788870079867,
 'LinearRegression_mean_absolute_error_difference': 0.11430484366588346,
 'avg_r2_difference': -3.9807132414904363,
 'avg_explained_variance_difference': -3.9781404626179113,
 'avg_mean_squared_error_difference': 0.046606404361905364,
 'avg_mean_absolute_error_difference': 0.11752566353633044,
 'MLPRegressor_r2_difference': -6.63321190611146,
 'MLPRegressor_explained_variance_difference': -7.109344159976527,
 'MLPRegressor_mean_squared_error_difference': 0.0823367448908961,
 'MLPRegressor_mean_absolute_error_difference': 0.20550288928659344,
 'XGBRegressor_r2_difference': -2.398171842098236,
 'XGBRegressor_explained_variance_difference': -2.3976160287857056,
 'XGBRegressor_mean_squared_error_difference': 0.02594895474612713,
 'XGBRegressor_mean_absolute_error_difference': 0.0789480097591877,